# CopyLane 판정 인코더 — 코랩 드라이런 (v2 · 데이터 확대판)

**목적**: 실제 팀 골든셋(`golden_sample`)이 아직 0건이라 진짜 파인튼닝은 못 한다.
이 노트북은 **파이프라인이 실제로 도는지**를 소량 더미 데이터로 미리 검증하는 것이 목적이다.

**v1(20개·5epoch) 대비 바뀐 점** — v1 실행 결과 위법 유형 헤드가 클래스 불균형
(위반없음 60%) 때문에 테스트 문장 3개 전부 `(없음)`으로만 예측했다. v2는 표본을
60개로 늘리고 클래스 균형을 조정, epoch도 5→20으로 늘렸다. train/dev 분리도 추가했다.

- 백본: `beomi/kcbert-base` (Apache 2.0, D-70 확정)
- 태스크: 카테고리 판별(4클래스) · **위법 유형(다중 라벨, D-54·D-65)** · 위험도(5값, 순서형) · 질의 의도(5클래스) — 멀티태스크 헤드 (기획서 5-1절)
- 데이터: **직접 작성한 예시 문장 60개** — 실제 성능을 보는 게 아니라 코드가 안 깨지고 도는지, loss가 내려가는지, 체크포인트·추론이 되는지만 확인한다.
- 근거: `docs/00_설계결정기록.md` D-08·D-65·D-70·D-131, `app/contracts.py`(Violation enum), `scripts/collect.py`(VIOLATION_TYPES/CANDIDATE_TYPES), `docs/01_기획/02_프로젝트기획서.md` 5-1·5-5절

⚠️ 여기서 나오는 정확도·loss 숫자는 **의미 없다** — 표본이 수십 건뿐이라 의도적으로 과적합시켜서 "파이프라인 자체가 정상 동작하는가"만 본다. 실제 성능 검증은 팀 골든셋 공급 이후.

## 0. 환경 설정 — 코랩에서 실행

In [1]:
!pip install -q transformers torch scikit-learn

In [2]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
import random

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device: cuda


## 1. 위법 유형 라벨 체계 — `app/contracts.py`의 `Violation` 그대로

기존 노트북은 위법 유형을 반영하지 않고 `risk`(위험도 5값)만 뒀었다. 실제 스키마는
`app/contracts.py`의 `Violation` enum(팀장 소유, `db/schema.sql`의 `violation_t`와 동일해야 함 — D-54)
이 정답이고, **다중 라벨**이다(한 문장에 여러 유형이 동시에 걸릴 수 있음).

🚨 **11종 전부가 지금 학습 대상은 아니다.** `scripts/collect.py`의 주석(D-65):

> 뒤 다섯은 **편입 후보**다 — 인코더가 예측하는 **확정 클래스는 `VIOLATION_TYPES` 6종**이고,
> **승격 판정일은 2026-09-17**(오늘)이다.

판정 규칙은 D-40·D-65 기준 유형별 **30건 미만 = 측정 불가**다. 첨부된 실측 차트
(`test_sentence` 206건, 유형별 평가 가능 하한 30건)가 이 판정의 근거 데이터로 보인다 —
`거짓_과장`(73)·`소비자_기만`(67)만 30건을 넘고, 나머지는 전부 하한 미달(부당_비교광고 4·비방광고 2·나머지 0).

In [3]:
# app/contracts.py 의 Violation enum 그대로 — db/schema.sql violation_t 와 동일해야 함 (D-54)

# D-65 확정 6종 — scripts/collect.py VIOLATION_TYPES. 인코더가 실제로 예측하는 클래스.
VIOLATION_TYPES = [
    "질병_예방치료_표방",
    "건강기능식품_오인",
    "의약품_오인",
    "거짓_과장",
    "소비자_기만",
    "후기_체험기_기만",
]

# 편입 후보 5종 — scripts/collect.py CANDIDATE_TYPES. 아직 정식 학습 클래스가 아니다.
# (2026-09-17 실측: 거짓_과장 73 · 소비자_기만 67 만 30건 하한 통과, 나머지는 하한 미달)
CANDIDATE_TYPES = [
    "추천_보증_뒷광고",
    "부당_비교광고",      # 실측 4건 — 하한 미달
    "비방광고",           # 실측 2건 — 하한 미달
    "실증책임_위반",
    "기능성화장품_오인",
]

ALL_VIOLATION_TYPES = VIOLATION_TYPES + CANDIDATE_TYPES
MIN_SAMPLES = 30  # D-40 — 이 미만은 '측정 불가'로 취급, Accuracy로 읽지 않는다

print(f"확정 클래스(학습 대상): {len(VIOLATION_TYPES)}종")
print(f"편입 후보(참고용, 이번 드라이런 제외): {len(CANDIDATE_TYPES)}종")

확정 클래스(학습 대상): 6종
편입 후보(참고용, 이번 드라이런 제외): 5종


⚠️ **이번 드라이런은 `VIOLATION_TYPES` 확정 6종만 학습한다.** 편입 후보 5종은 D-65 판정
(오늘) 이후 팀 결정에 따라 편입되거나 「측정 불가 유형」으로 남는다 — 지금 섞어서 학습하면
나중에 클래스 집합이 바뀔 때 다시 갈아엎어야 한다(D-08 「학습 중 클래스가 바뀌면 처음부터
다시」).

## 2. 더미 예시 데이터 (확장판 — 60개)

실제 결함 주입 골든셋(D-25·D-74, TRANSFORM_RULES T1~T10)을 흉내 낸 **직접 작성한 샘플**이다.
실제 데이터가 아니므로 라벨 분포·표현이 거칠다 — 파이프라인 검증용으로만 쓴다.

**1차 드라이런(20개) 대비 바뀐 점** — 지난 실행에서 위법 유형 헤드가 클래스 불균형
(위반없음 12/20 = 60%) 때문에 3문장 전부 `(없음)`으로만 예측하는 문제가 있었다. 이번엔:

- 20 → **60개**로 표본 확대
- 위반없음 비율을 60% → 약 30%로 낮춤 — 나머지는 6개 확정 유형에 최소 6건씩 고르게 배분
- 유형별 단독 사례뿐 아니라 **다중 라벨 조합**(2~3개 유형 동시)도 늘림 — 실제 위반 문구는 여러 유형이 겹치는 경우가 흔함(D-74 변환 규칙표 참고)
- **train/dev 분리 추가** — 마지막 12건(20%)을 dev로 떼어 「학습에 없던 문장에도 되는가」를 살짝 볼 수 있게 함(완전한 검증은 아니고 참고용)

라벨 스키마 (기획서 5-1절 요약):
- `category`: 0=일반, 1=식품, 2=건기식, 3=화장품
- `violations`: `VIOLATION_TYPES` 6종에 대한 **다중 라벨**(멀티핫) — 위 1장 참고
- `risk`: 0=특이사항없음 1=주의 2=업무정지위험 3=과징금위험 4=형사위험 (순서형, `violations`와 별개 축)
- `intent`: 0=문구검수 1=법령조회 2=사례검색 3=대체문구요청 4=복합

In [4]:
# (text, category, violations(멀티핫 6자리), risk, intent)
# violations 순서 = VIOLATION_TYPES 순서: [질병_예방치료_표방, 건강기능식품_오인, 의약품_오인, 거짓_과장, 소비자_기만, 후기_체험기_기만]
DUMMY_SAMPLES = [
    # --- 위반없음 (18건 ≈ 30%) ---
    ("이 크림은 피부 진정에 도움을 줄 수 있다고 알려져 있습니다", 3, [0,0,0,0,0,0], 0, 0),
    ("제26조가 어떤 내용인가요", 0, [0,0,0,0,0,0], 0, 1),
    ("비슷한 위반 사례가 있을까요", 0, [0,0,0,0,0,0], 0, 2),
    ("이 문구 대신 쓸 표현을 추천해주세요", 0, [0,0,0,0,0,0], 0, 3),
    ("이거 왜 위반인지랑 고친 문구도 같이 알려주세요", 0, [0,0,0,0,0,0], 0, 4),
    ("임상시험으로 효과가 입증된 성분 함유", 2, [0,0,0,0,0,0], 1, 0),          # 경계 클래스(실증 있음)
    ("성분표만 보면 이 제품이 합법인지 알 수 있나요", 0, [0,0,0,0,0,0], 0, 1),
    ("이런 표현으로 바꾸면 안전한가요 검토도 부탁해요", 0, [0,0,0,0,0,0], 0, 4),
    ("영양성분 기준에 맞게 표시했습니다", 1, [0,0,0,0,0,0], 0, 0),
    ("기능성 원료 함유로 도움을 줄 수 있습니다", 2, [0,0,0,0,0,0], 0, 0),      # 인정 기능성 문구 그대로
    ("과징금 처분 기준이 궁금합니다", 0, [0,0,0,0,0,0], 0, 1),
    ("작년에 비슷한 문구로 처분받은 사례 있나요", 0, [0,0,0,0,0,0], 0, 2),
    ("보존료 함량을 제품 라벨에 기재했습니다", 1, [0,0,0,0,0,0], 0, 0),
    ("자외선 차단 효과가 있는 제품입니다 (SPF30 실증 완료)", 3, [0,0,0,0,0,0], 0, 0),
    ("이 제품의 알레르기 유발 성분을 알려주세요", 0, [0,0,0,0,0,0], 0, 1),
    ("경쟁 제품 사례도 같이 검토해줄 수 있나요", 0, [0,0,0,0,0,0], 0, 2),
    ("이 문구는 표시광고법상 문제 없나요", 0, [0,0,0,0,0,0], 0, 0),
    ("할인 행사 안내 문구인데 검토 부탁드립니다", 1, [0,0,0,0,0,0], 0, 0),

    # --- 질병_예방치료_표방 단독/조합 (8건) ---
    ("이 크림 바르면 아토피가 완치됩니다", 3, [1,0,0,0,0,0], 4, 0),
    ("매일 드시면 당뇨가 낫습니다", 1, [1,0,0,0,0,0], 4, 0),
    ("이 성분이 암 예방에 직접적 효과가 있습니다", 2, [1,0,0,1,0,0], 4, 0),   # +거짓_과장
    ("고혈압 치료에 탁월한 건강기능식품입니다", 2, [1,1,0,0,0,0], 4, 0),      # +건강기능식품_오인
    ("이 로션으로 아토피 피부염을 근본 치료하세요", 3, [1,0,0,0,0,0], 4, 0),
    ("관절염 통증이 완전히 사라지는 영양제", 2, [1,0,0,1,0,0], 4, 0),        # +거짓_과장
    ("불면증을 치료해주는 수면 유도 성분", 1, [1,0,0,0,0,0], 4, 0),
    ("비염 예방과 치료에 효과적인 스프레이", 1, [1,0,0,0,0,0], 4, 0),

    # --- 건강기능식품_오인 단독/조합 (7건) ---
    ("이 영양제 드시면 면역력이 확실히 증진됩니다", 2, [0,1,0,1,0,0], 2, 0),  # +거짓_과장(T1)
    ("이 제품은 건강기능식품 인증 성분과 동일한 효과", 1, [0,1,0,0,0,0], 2, 0),
    ("일반식품이지만 건기식만큼 면역에 좋습니다", 1, [0,1,0,0,0,0], 2, 0),
    ("이 음료는 기능성 원료 함량이 건기식 기준을 충족합니다", 1, [0,1,0,0,0,0], 1, 0),
    ("건강기능식품 수준의 항산화 효과를 가진 일반식품", 1, [0,1,0,1,0,0], 2, 0),  # +거짓_과장
    ("이 제품 섭취만으로 건기식 인정 기능성을 대체합니다", 1, [0,1,0,0,0,0], 2, 0),
    ("체지방 감소 효과가 건강기능식품과 동등합니다", 1, [0,1,0,1,0,0], 2, 0),  # +거짓_과장

    # --- 의약품_오인 단독/조합 (7건) ---
    ("이 제품은 국내 유일 특허 치료제입니다", 1, [0,0,1,1,0,0], 4, 0),        # +거짓_과장(T4)
    ("처방전 없이도 살 수 있는 의약품급 효능", 2, [0,0,1,0,0,0], 4, 0),
    ("이 연고는 상처 치료제와 동일한 성분입니다", 3, [0,0,1,0,0,0], 4, 0),
    ("항염 치료 효과가 있는 특효 성분 함유", 2, [0,0,1,1,0,0], 4, 0),        # +거짓_과장
    ("약국에서 파는 소화제만큼 효과가 확실합니다", 1, [0,0,1,0,0,0], 3, 0),
    ("이 크림은 피부과 처방약 수준의 치료 효과", 3, [0,0,1,0,0,0], 4, 0),
    ("감기약 대신 먹어도 되는 즉효 성분", 1, [0,0,1,0,0,0], 4, 0),

    # --- 거짓_과장 단독/조합 (8건) ---
    ("다들 드시고 확실히 효과 보셨어요", 1, [0,0,0,1,1,0], 1, 0),            # +소비자_기만(T9)
    ("경쟁 제품보다 훨씬 효과가 좋습니다", 1, [0,0,0,1,0,0], 2, 0),
    ("세포 재생에 탁월한 효과가 있는 안티에이징 크림", 3, [0,0,0,1,0,0], 2, 0),
    ("먹기만 하면 살이 쫙 빠집니다", 1, [0,0,0,1,1,0], 4, 0),                # +소비자_기만
    ("100% 완치 보장, 유일무이한 효능", 2, [0,0,0,1,0,0], 3, 0),
    ("업계 최고 효과, 타사 제품은 비교 불가", 1, [0,0,0,1,0,0], 2, 0),
    ("놀라운 즉각 효과, 사용 즉시 체감", 3, [0,0,0,1,0,0], 1, 0),
    ("과학적으로 100% 입증된 최고의 성분", 2, [0,0,0,1,0,0], 2, 0),

    # --- 소비자_기만 단독/조합 (6건) ---
    ("이 제품 쓰고 인생이 바뀌었어요", 3, [0,0,0,0,1,0], 1, 0),
    ("고객님들이 재구매율 100%라고 극찬한 제품", 1, [0,0,0,1,1,0], 2, 0),    # +거짓_과장
    ("원가 그대로 드리는 마지막 특가입니다", 1, [0,0,0,0,1,0], 1, 0),
    ("실제로 다 나았다는 후기가 압도적입니다", 2, [0,0,0,0,1,0], 1, 0),
    ("전문가도 인정한 효과, 모두가 만족했습니다", 1, [0,0,0,1,1,0], 2, 0),   # +거짓_과장
    ("이 결과는 개인차 없이 누구나 동일합니다", 2, [0,0,0,0,1,0], 1, 0),

    # --- 후기_체험기_기만 단독/조합 (6건) ---
    ("체험단 후기입니다 정말 만족스러워요 (광고 표시 없음)", 1, [0,0,0,0,0,1], 1, 0),
    ("실사용 후기 이렇게 좋을 줄 몰랐어요 (협찬 미표시)", 3, [0,0,0,0,0,1], 1, 0),
    ("내돈내산이라고 적었지만 사실 협찬받았습니다", 1, [0,0,0,0,1,1], 1, 0), # +소비자_기만
    ("블로거 체험단 모집 후기, 대가 지급 사실 비공개", 2, [0,0,0,0,0,1], 1, 0),
    ("인플루언서 추천이라고만 표시하고 광고임을 숨김", 1, [0,0,0,0,0,1], 1, 0),
    ("체험 후기 게시물인데 실제로는 유료 광고입니다", 3, [0,0,0,0,0,1], 1, 0),
]

print(f"샘플 수: {len(DUMMY_SAMPLES)}")

# 클래스 분포 확인 — 위반없음 비율이 낮아졌는지, 유형별로 6건 이상 확보됐는지
n_no_violation = sum(1 for s in DUMMY_SAMPLES if sum(s[2]) == 0)
print(f"위반없음: {n_no_violation}건 ({n_no_violation/len(DUMMY_SAMPLES)*100:.0f}%)")
for i, vtype in enumerate(VIOLATION_TYPES):
    count = sum(1 for s in DUMMY_SAMPLES if s[2][i] == 1)
    print(f"  {vtype}: {count}건")

texts, cat_labels, violation_labels, risk_labels, intent_labels = zip(*DUMMY_SAMPLES)

# train/dev 분리 — 마지막 12건(20%)을 dev로. 셔플하지 않는다(재현성 + 유형별 분포 유지 목적).
N_DEV = 12
train_texts, dev_texts = texts[:-N_DEV], texts[-N_DEV:]
train_cat, dev_cat = cat_labels[:-N_DEV], cat_labels[-N_DEV:]
train_violation, dev_violation = violation_labels[:-N_DEV], violation_labels[-N_DEV:]
train_risk, dev_risk = risk_labels[:-N_DEV], risk_labels[-N_DEV:]
train_intent, dev_intent = intent_labels[:-N_DEV], intent_labels[-N_DEV:]
print(f"\ntrain: {len(train_texts)}건 · dev: {len(dev_texts)}건")

샘플 수: 60
위반없음: 18건 (30%)
  질병_예방치료_표방: 8건
  건강기능식품_오인: 8건
  의약품_오인: 7건
  거짓_과장: 17건
  소비자_기만: 9건
  후기_체험기_기만: 6건

train: 48건 · dev: 12건


## 3. 토크나이저 · 백본 로드 (kcbert-base)

In [5]:
MODEL_NAME = "beomi/kcbert-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
backbone = AutoModel.from_pretrained(MODEL_NAME)
print(backbone.config)

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/250k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "directionality": "bidi",
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 300,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "pooler_fc_size": 768,
  "pooler_num_attention_heads": 12,
  "pooler_num_fc_layers": 3,
  "pooler_size_per_head": 128,
  "pooler_type": "first_token_transform",
  "tie_word_embeddings": true,
  "transformers_version": "5.16.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30000
}



## 4. 멀티태스크 헤드 정의

기획서 5-1절 태스크 표를 그대로 반영 — 카테고리(4클래스 분류) · **위법 유형(6클래스 다중 라벨)**
· 위험도(5차원 순서형) · 의도(5클래스 분류)를 같은 kcbert-base 인코더 위에 헤드 4개로 얹는다.

위법 유형은 **다중 라벨**이라 softmax+CrossEntropy가 아니라 **sigmoid+BCE**로 학습한다
(한 문장이 거짓_과장이면서 동시에 소비자_기만일 수 있음 — 위 더미 데이터 5번·17번 참고).

실제 근거 스팬 헤드(BIO, D-131)는 이번 드라이런에서는 생략한다 — 토큰 분류는 시퀀스 라벨링이 따로 필요해서
별도 검증이 맞고, 여기서는 문장 단위 분류/회귀 파이프라인이 도는지만 먼저 확인한다.

In [6]:
class JudgeEncoder(nn.Module):
    """기획서 5-1절 멀티태스크 헤드 — 카테고리 / 위법유형(다중라벨) / 위험도(순서형) / 의도."""
    def __init__(self, backbone, n_category=4, n_violation=6, n_risk=5, n_intent=5):
        super().__init__()
        self.backbone = backbone
        hidden = backbone.config.hidden_size
        self.category_head = nn.Linear(hidden, n_category)
        self.violation_head = nn.Linear(hidden, n_violation)  # 다중 라벨 — sigmoid+BCE
        self.risk_head = nn.Linear(hidden, n_risk)             # 순서형이지만 드라이런에서는 분류로 단순화
        self.intent_head = nn.Linear(hidden, n_intent)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]  # [CLS]
        return {
            "category": self.category_head(cls),
            "violation": self.violation_head(cls),  # 로짓 — 학습 loss에서 sigmoid 적용
            "risk": self.risk_head(cls),
            "intent": self.intent_head(cls),
        }

model = JudgeEncoder(backbone).to(device)
print(model)

JudgeEncoder(
  (backbone): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30000, 768, padding_idx=0)
      (position_embeddings): Embedding(300, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_

## 5. 데이터셋 · 데이터로더 (train/dev 분리)

In [7]:
from torch.utils.data import Dataset, DataLoader

class DummyJudgeDataset(Dataset):
    def __init__(self, texts, cat, violations, risk, intent, tokenizer, max_len=64):
        self.texts, self.cat, self.violations = texts, cat, violations
        self.risk, self.intent = risk, intent
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, padding="max_length",
            max_length=self.max_len, return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "category": torch.tensor(self.cat[idx], dtype=torch.long),
            "violation": torch.tensor(self.violations[idx], dtype=torch.float),  # 멀티핫 — BCE용 float
            "risk": torch.tensor(self.risk[idx], dtype=torch.long),
            "intent": torch.tensor(self.intent[idx], dtype=torch.long),
        }

train_dataset = DummyJudgeDataset(train_texts, train_cat, train_violation, train_risk, train_intent, tokenizer)
dev_dataset = DummyJudgeDataset(dev_texts, dev_cat, dev_violation, dev_risk, dev_intent, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=8, shuffle=False)
print(f"train 배치 수: {len(train_loader)} · dev 배치 수: {len(dev_loader)}")

train 배치 수: 6 · dev 배치 수: 2


## 6. 학습 루프 (드라이런, epoch 확대)

1차 드라이런은 20개·5 epoch였는데 위법 유형(다중 라벨) 헤드가 다수 클래스(위반없음)로
쏠려 3문장 전부 `(없음)`을 예측했다. 이번엔 60개(위반없음 비율 축소)·**20 epoch**로
늘려서 같은 문제가 완화되는지 본다.

매 5 epoch마다 dev 세트로 간단 점검도 같이 한다 — 학습에 없던 문장에 대한 결과라
참고용이지 정식 평가(D-77)는 아니다.

위법 유형만 손실 함수가 다르다(BCEWithLogitsLoss — 다중 라벨) — 나머지 세 헤드는 CrossEntropyLoss.

In [8]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
ce_loss_fn = nn.CrossEntropyLoss()
bce_loss_fn = nn.BCEWithLogitsLoss()  # 위법 유형 다중 라벨용

EPOCHS = 20  # 드라이런용 — 실제 학습에서는 데이터·검증 세트 기준으로 다시 정한다

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            cat = batch["category"].to(device)
            violation = batch["violation"].to(device)
            risk = batch["risk"].to(device)
            intent = batch["intent"].to(device)

            if train:
                optimizer.zero_grad()
            out = model(input_ids, attention_mask)
            loss = (
                ce_loss_fn(out["category"], cat)
                + bce_loss_fn(out["violation"], violation)
                + ce_loss_fn(out["risk"], risk)
                + ce_loss_fn(out["intent"], intent)
            )
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
    return total_loss / len(loader)

history_train, history_dev = [], []
for epoch in range(EPOCHS):
    train_loss = run_epoch(train_loader, train=True)
    history_train.append(train_loss)
    if (epoch + 1) % 5 == 0 or epoch == EPOCHS - 1:
        dev_loss = run_epoch(dev_loader, train=False)
        history_dev.append((epoch + 1, dev_loss))
        print(f"epoch {epoch+1}/{EPOCHS}  train_loss={train_loss:.4f}  dev_loss={dev_loss:.4f}")
    else:
        print(f"epoch {epoch+1}/{EPOCHS}  train_loss={train_loss:.4f}")

print("\n학습 루프 정상 종료 — train loss 추이:", [round(x, 4) for x in history_train])
print("dev loss 체크포인트:", history_dev)

epoch 1/20  train_loss=4.6644
epoch 2/20  train_loss=3.1214
epoch 3/20  train_loss=2.3219
epoch 4/20  train_loss=1.6920
epoch 5/20  train_loss=1.2178  dev_loss=4.2241
epoch 6/20  train_loss=0.9581
epoch 7/20  train_loss=0.7225
epoch 8/20  train_loss=0.5904
epoch 9/20  train_loss=0.5044
epoch 10/20  train_loss=0.4044  dev_loss=4.5202
epoch 11/20  train_loss=0.3602
epoch 12/20  train_loss=0.3235
epoch 13/20  train_loss=0.2965
epoch 14/20  train_loss=0.2649
epoch 15/20  train_loss=0.2413  dev_loss=5.0555
epoch 16/20  train_loss=0.2368
epoch 17/20  train_loss=0.2211
epoch 18/20  train_loss=0.2034
epoch 19/20  train_loss=0.1922
epoch 20/20  train_loss=0.1750  dev_loss=5.2696

학습 루프 정상 종료 — train loss 추이: [4.6644, 3.1214, 2.3219, 1.692, 1.2178, 0.9581, 0.7225, 0.5904, 0.5044, 0.4044, 0.3602, 0.3235, 0.2965, 0.2649, 0.2413, 0.2368, 0.2211, 0.2034, 0.1922, 0.175]
dev loss 체크포인트: [(5, 4.224058747291565), (10, 4.520235180854797), (15, 5.0554585456848145), (20, 5.269552946090698)]


## 7. 체크포인트 저장 확인

실제 파인튜닝에서도 같은 방식으로 저장되는지 확인 — 코랩이면 `/content/checkpoints/`,
드라이브 마운트 시 `/content/drive/MyDrive/...`로 경로만 바꾸면 된다.

In [9]:
import os

CKPT_DIR = "/content/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

ckpt_path = os.path.join(CKPT_DIR, "judge_encoder_dryrun.pt")
torch.save(model.state_dict(), ckpt_path)
print("저장 완료:", ckpt_path, f"({os.path.getsize(ckpt_path) / 1e6:.1f} MB)")

저장 완료: /content/checkpoints/judge_encoder_dryrun.pt (435.8 MB)


## 8. 추론 확인 — dev 세트(학습에 안 쓴 문장) + 새 문장

1차 드라이런은 새로 지어낸 문장 3개만 봤다. 이번엔 **dev 세트(학습에서 뺀 12건)** 로
먼저 확인하고, 그 다음 전혀 새로운 문장으로도 확인한다 — dev 세트는 최소한 같은 스타일로
쓰인 문장이라 완전히 새 표현보다는 맞을 가능성이 높아야 정상이다.

In [10]:
CATEGORY_LABELS = ["일반", "식품", "건기식", "화장품"]
RISK_LABELS = ["특이사항없음", "주의", "업무정지위험", "과징금위험", "형사위험"]
INTENT_LABELS = ["문구검수", "법령조회", "사례검색", "대체문구요청", "복합"]
VIOLATION_THRESHOLD = 0.5  # 다중 라벨 — sigmoid 출력이 이 값 넘으면 해당 유형으로 판정

def predict(sent):
    enc = tokenizer(sent, truncation=True, padding="max_length", max_length=64, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model(enc["input_ids"], enc["attention_mask"])
    cat_pred = out["category"].argmax(-1).item()
    risk_pred = out["risk"].argmax(-1).item()
    intent_pred = out["intent"].argmax(-1).item()
    violation_probs = torch.sigmoid(out["violation"]).squeeze(0).tolist()
    violation_preds = [VIOLATION_TYPES[i] for i, p in enumerate(violation_probs) if p >= VIOLATION_THRESHOLD]
    return cat_pred, violation_preds, risk_pred, intent_pred

model.eval()
print("=== dev 세트(학습에 안 쓴 12건) — 정답과 비교 ===\n")
for i, sent in enumerate(dev_texts):
    cat_pred, violation_preds, risk_pred, intent_pred = predict(sent)
    gold_violations = [VIOLATION_TYPES[j] for j, v in enumerate(dev_violation[i]) if v == 1]
    match = "O" if set(violation_preds) == set(gold_violations) else "X"
    print(f"[{match}] 문장: {sent}")
    print(f"    예측: 카테고리={CATEGORY_LABELS[cat_pred]} · 위법유형={violation_preds or '(없음)'} · 위험도={RISK_LABELS[risk_pred]} · 의도={INTENT_LABELS[intent_pred]}")
    print(f"    정답: 카테고리={CATEGORY_LABELS[dev_cat[i]]} · 위법유형={gold_violations or '(없음)'} · 위험도={RISK_LABELS[dev_risk[i]]} · 의도={INTENT_LABELS[dev_intent[i]]}")
    print()

print("\n=== 완전히 새로운 문장 (학습·dev 어디에도 없던 표현) ===\n")
test_sentences = [
    "이 영양제는 면역력에 도움을 줄 수 있다고 알려져 있습니다",
    "이 크림 바르면 주름이 완전히 사라집니다",
    "이 표현 대신 뭘 쓰면 좋을까요",
]
for sent in test_sentences:
    cat_pred, violation_preds, risk_pred, intent_pred = predict(sent)
    print(f"문장: {sent}")
    print(f"  → 카테고리={CATEGORY_LABELS[cat_pred]} · 위험도={RISK_LABELS[risk_pred]} · 의도={INTENT_LABELS[intent_pred]}")
    print(f"  → 위법 유형: {violation_preds if violation_preds else '(없음)'}")
    print()

=== dev 세트(학습에 안 쓴 12건) — 정답과 비교 ===

[X] 문장: 이 제품 쓰고 인생이 바뀌었어요
    예측: 카테고리=식품 · 위법유형=(없음) · 위험도=특이사항없음 · 의도=문구검수
    정답: 카테고리=화장품 · 위법유형=['소비자_기만'] · 위험도=주의 · 의도=문구검수

[X] 문장: 고객님들이 재구매율 100%라고 극찬한 제품
    예측: 카테고리=식품 · 위법유형=(없음) · 위험도=업무정지위험 · 의도=문구검수
    정답: 카테고리=식품 · 위법유형=['거짓_과장', '소비자_기만'] · 위험도=업무정지위험 · 의도=문구검수

[X] 문장: 원가 그대로 드리는 마지막 특가입니다
    예측: 카테고리=식품 · 위법유형=(없음) · 위험도=특이사항없음 · 의도=문구검수
    정답: 카테고리=식품 · 위법유형=['소비자_기만'] · 위험도=주의 · 의도=문구검수

[X] 문장: 실제로 다 나았다는 후기가 압도적입니다
    예측: 카테고리=식품 · 위법유형=(없음) · 위험도=특이사항없음 · 의도=문구검수
    정답: 카테고리=건기식 · 위법유형=['소비자_기만'] · 위험도=주의 · 의도=문구검수

[X] 문장: 전문가도 인정한 효과, 모두가 만족했습니다
    예측: 카테고리=식품 · 위법유형=['거짓_과장'] · 위험도=업무정지위험 · 의도=문구검수
    정답: 카테고리=식품 · 위법유형=['거짓_과장', '소비자_기만'] · 위험도=업무정지위험 · 의도=문구검수

[X] 문장: 이 결과는 개인차 없이 누구나 동일합니다
    예측: 카테고리=일반 · 위법유형=(없음) · 위험도=특이사항없음 · 의도=문구검수
    정답: 카테고리=건기식 · 위법유형=['소비자_기만'] · 위험도=주의 · 의도=문구검수

[X] 문장: 체험단 후기입니다 정말 만족스러워요 (광고 표시 없음)
    예측: 카테고리=식품 · 위법유형=(없음) · 위험도=특이사항없음 · 의도=문구검수
    정답: 카테고리=식품 · 위법유형=['후기

## 9. 정리 — 이 드라이런이 확인한 것 / 안 한 것

**확인한 것 (파이프라인 정상 동작)**
- kcbert-base 로드 및 forward pass 정상
- 멀티태스크 헤드(카테고리/**위법유형(다중라벨)**/위험도/의도) 학습 루프가 에러 없이 돎
- 위법 유형만 다른 손실 함수(BCE, 다중 라벨) 쓰는 구조가 정상 작동
- 표본 20→60개, 클래스 불균형 완화(위반없음 60%→30%), epoch 5→20으로 확대
- train/dev 분리로 dev 세트(학습에 없던 문장) 결과도 참고용으로 확인
- 체크포인트 저장/로드 가능

**확인하지 않은 것 (실제 성능 관련 — 팀 골든셋 공급 후 필요)**
- 진짜 일반화 성능 — dev 12건은 참고용이지 정식 test_holdout이 아니다(D-25 「주입본은 절대 test_holdout에 넣지 않는다」와 다른 목적)
- 근거 스팬 헤드(BIO, D-131) — 토큰 분류라 시퀀스 라벨링 별도 구현 필요
- 위험도 순서형 특성 반영 (여기선 단순 분류로 처리 — 실제로는 QWK 기준 순서형 회귀/제약 필요, D-131)
- **편입 후보 5종(추천_보증_뒷광고·부당_비교광고·비방광고·실증책임_위반·기능성화장품_오인)** — D-65 판정(2026-09-17) 결과에 따라 클래스 집합이 바뀔 수 있어 이번 드라이런에서는 제외
- Span F1 · MAE · QWK · selective risk 등 D-77 공식 평가지표
- DAPT(도메인 적응 사전학습) 적용 여부 비교
- kcbert vs KcELECTRA 베이스라인 비교 (D-70)

### 다음 단계
1. **D-65 판정 결과 확인** (2026-09-17) — 편입 후보 중 30건 이상인 유형이 있으면 `VIOLATION_TYPES`에 추가 반영
2. `preprocess/golden.py` 실행 가능해지면(팀 골든셋 공급 후) 이 노트북의 `DUMMY_SAMPLES`를 실제 `golden_sample` 데이터로 교체
3. 근거 스팬 헤드 추가 (BIO 태깅)
4. 진짜 train/dev/test_holdout 분리 + D-77 평가지표(유형별 분리 Recall/Precision/F1) 적용